# Análisis de resultados

Carga las corridas (de Drive o de `resultados.zip`), y corre `analysis.make_all`, que genera todas las figuras (`docs/figures/`) y tablas (`analysis/results/`) de los experimentos que ya tienen datos. Los que faltan se saltan.

In [ ]:
import getpass, os
try:
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
except Exception:
    os.environ["GITHUB_TOKEN"] = getpass.getpass("GITHUB_TOKEN: ")
REPO_DIR = "/content/ProyectoFinalAR"
BRANCH = "main"
if not os.path.isdir(REPO_DIR + "/.git"):
    !git clone --branch {BRANCH} https://$GITHUB_TOKEN@github.com/Alexistodj124/ProyectoFinalAR.git {REPO_DIR}
    !git -C {REPO_DIR} remote set-url origin https://github.com/Alexistodj124/ProyectoFinalAR.git
%cd {REPO_DIR}
!pip install -q numpy pandas scipy matplotlib pyyaml
os.environ["PYTHONPATH"] = REPO_DIR

## Cargar las corridas
Elige la fuente: **Drive** (la misma carpeta que usó el cuaderno de entrenamiento) o **zip** (sube `resultados.zip` con el panel de archivos o `files.upload()`).

In [ ]:
SOURCE = "drive" #@param ["drive", "zip"]
DRIVE_DIR = "/content/drive/MyDrive/ProyectoFinalAR/runs" #@param {type:"string"}
ZIP_PATH = "/content/resultados.zip" #@param {type:"string"}
if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p runs && rsync -a --exclude 'tb/' {DRIVE_DIR}/ runs/
else:
    if not os.path.exists(ZIP_PATH):
        from google.colab import files
        up = files.upload(); ZIP_PATH = "/content/" + next(iter(up))
    !unzip -oq {ZIP_PATH} -d .
!find runs -maxdepth 3 -name DONE | sort | head -50

## Generar figuras y tablas

In [ ]:
from analysis.make_all import make_all
made = make_all()

## Ver las figuras

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("docs/figures/*.png")):
    print(p); display(Image(p))

## Ver las tablas (Markdown)

In [ ]:
from IPython.display import Markdown, display
for p in sorted(glob.glob("analysis/results/*.md")):
    display(Markdown(open(p).read()))

## Descargar figuras y tablas

In [ ]:
!zip -qr analisis.zip docs/figures analysis/results
from google.colab import files
files.download("analisis.zip")